# Voxtral Audio Cookbook

Voxtral is Mistral's first native audio model family. Unlike transcription-only tools, Voxtral **understands** audio — you can transcribe, ask questions, and extract structured data all in a single API call, without any intermediate text step.

This notebook walks through the core use cases:

- Basic transcription (dedicated endpoint + chat completions)
- Audio Q&A (direct questions on audio)
- Structured extraction (JSON output from audio)
- Speaker diarization
- Model comparison: `voxtral-mini-latest` vs `voxtral-small-latest`

*Author: [Valentin Dugay](https://github.com/valdugay)*

## Setup

In [1]:
%pip install mistralai python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import base64
import json
import time
from dotenv import load_dotenv
from mistralai.client import Mistral

load_dotenv()

api_key = os.environ["MISTRAL_KEY"]
client = Mistral(api_key=api_key)

### Load the audio file

We use a single audio file throughout the notebook — a short podcast-style conversation between two speakers discussing AI and audio processing. We load it once and reuse it across all sections.

Voxtral accepts audio in two ways depending on the endpoint:
- **Dedicated transcription endpoint** (`/v1/audio/transcriptions`): pass the file as a binary stream
- **Chat completions**: pass the file as a base64-encoded string inside the message content

In [ ]:
import requests

audio_url = "https://raw.githubusercontent.com/mistralai/cookbook/main/data/audio.mp3"
with open("audio.mp3", "wb") as f:
    f.write(requests.get(audio_url).content)
print("Audio downloaded.")

Audio downloaded.


In [3]:
AUDIO_FILE = "audio.mp3"

with open(AUDIO_FILE, "rb") as f:
    audio_bytes = f.read()

audio_base64 = base64.b64encode(audio_bytes).decode("utf-8")

print(f"Audio loaded: {len(audio_bytes) / 1024:.1f} KB")

Audio loaded: 931.1 KB


## Basic Transcription

The simplest use case: convert audio to text. There are two ways to do this with Voxtral, each suited to different situations.

### Option 1 — Dedicated transcription endpoint

The `/v1/audio/transcriptions` endpoint is optimized purely for transcription. It's the fastest and cheapest option when you only need the text.

In [4]:
with open(AUDIO_FILE, "rb") as f:
    transcription = client.audio.transcriptions.complete(
        model="voxtral-mini-latest",
        file={"content": f, "file_name": AUDIO_FILE},
    )

print(transcription.text)

So let's talk about where AI and audio are actually heading. A year ago, the pipeline was always the same. Transcribe first, then pass the text to a language model. But it lost a lot along the way. Tone, hesitation, emphasis. All of that just disappears the moment you convert speech to text. And what's exciting now is that some models can take audio directly as input. You skip the transcription step entirely and ask questions straight away. It changes a lot for things like meeting summarization. Instead of running Whisper, cleaning the transcript, then prompting a model, you just send the recording and ask for the action. And the model actually heard the meeting, so it can pick up on who sounded uncertain. It's also diarization now, built into some of these APIs. Speaker labeling without any external tooling. The APIs are finally catching up to what people are doing. All right, that's a wrap for today. Thanks for listening.


### Option 2 — Chat completions with audio input

Voxtral can also receive audio directly inside a chat message. This gives you more control: system prompts, structured output, and Q&A all in the same call. Here we use it for a simple transcription to show the format.

In [5]:
response = client.chat.complete(
    model="voxtral-small-latest",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "input_audio", "input_audio": audio_base64},
                {"type": "text", "text": "Transcribe this audio."},
            ],
        }
    ],
)

print(response.choices[0].message.content)

So let's talk about where AI and audio are actually heading. A year ago, the pipeline was always the same. Transcribe first, then pass the text to a language model. But it lost a lot along the way. Tone, hesitation, emphasis, all of that just disappears the moment you convert speech to text. And what's exciting now is that some models can take audio directly as input. You skip the transcription step entirely and ask for things like meeting summarization. Instead of running whisper, cleaning the transcript, then prompting a model, you just send the recording and ask for the action. And the model actually heard the meeting, so it can pick up on who sounded uncertain. Built into some of these APIs, speaker labeling without any external tooling. The APIs are finally catching up to what people need. All right, that's a wrap for today. Thanks for listening.


#### Considerations:
- **Endpoint choice**: The dedicated endpoint (`/v1/audio/transcriptions`) is optimized for pure transcription — lower latency and lower cost. Chat completions are more flexible but intended for when you need understanding, not just text.
- **Model choice**: `voxtral-mini-latest` runs the transcription-optimized variant and is the most cost-efficient option. Use `voxtral-small-latest` (via chat completions) when you need richer understanding on top of transcription.
- **Language**: You can pass `language="en"` to the transcription endpoint to improve accuracy when the language is known.

## Audio Q&A

A key differentiator from Whisper-style tools: Voxtral can answer questions **directly on the audio**, with no intermediate transcription step. The model hears the recording and responds.

This matters because it preserves information that transcription loses — tone, emphasis, hesitation, and speaker context.

In [6]:
def ask_about_audio(question: str) -> str:
    response = client.chat.complete(
        model="voxtral-small-latest",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "input_audio", "input_audio": audio_base64},
                    {"type": "text", "text": question},
                ],
            }
        ],
    )
    return response.choices[0].message.content

In [7]:
ask_about_audio("What is the main topic discussed in this recording?")

'The main topic discussed is the evolution of AI in handling audio data, specifically focusing on how new models can process audio directly without the need for transcription, and the benefits this brings to tasks like meeting summarization.'

In [8]:
ask_about_audio("What argument do the speakers make about the old transcription pipeline?")

'The speakers argue that the old transcription pipeline, which involved transcribing audio first and then passing the text to a language model, resulted in a loss of important elements such as tone, hesitation, and emphasis.'

In [9]:
ask_about_audio("How many speakers are there, and how would you describe each of their voices?")

"The audio features two speakers. The first speaker has a calm and informative tone, explaining the advancements in AI and audio processing. The second speaker interjects with brief comments, such as 'Right,' and 'That's a wrap for today,' indicating a more casual and conversational tone."

#### Considerations:
- **No pipeline needed**: Traditional setups run Whisper → clean transcript → LLM. With Voxtral, this becomes a single API call.
- **Context window**: Long recordings may need chunking. Keep individual audio files under the model's audio context limit.
- **System prompts**: You can add a `system` message alongside the audio to control the response style or domain.

## Structured Extraction

Voxtral can extract structured data from audio in a single call using JSON mode. No transcription step, no post-processing chain — just audio in, structured JSON out.

Here we extract a summary, key points, and speakers from the recording.

In [10]:
system_prompt = """Listen to the audio carefully and extract the following information. Return valid JSON only.
{
  "topic": "main subject of the conversation",
  "summary": "2-3 sentence summary",
  "key_points": ["point 1", "point 2", "..."],
  "speakers": [
    "Describe the voice, tone, and role of the FIRST speaker you hear — their speaking style, energy, and what they contribute to the conversation",
    "Describe the voice, tone, and role of the SECOND speaker you hear — their speaking style, energy, and what they contribute to the conversation"
  ],
  "conclusion": "how the conversation ends"
}"""

response = client.chat.complete(
    model="voxtral-small-latest",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "input_audio", "input_audio": audio_base64},
                {"type": "text", "text": "Analyze this recording."},
            ],
        },
    ],
    response_format={"type": "json_object"},
)

audio_data = json.loads(response.choices[0].message.content)
audio_data

{'topic': 'Advancements in AI and Audio Processing',
 'summary': 'The conversation discusses the evolution of AI in audio processing, highlighting the shift from transcription-based methods to direct audio input models, which preserve more nuanced information and improve tasks like meeting summarization.',
 'key_points': ['Traditional pipeline involved transcribing audio first, losing tone and emphasis.',
  'New models can take audio directly as input, skipping transcription.',
  'Direct audio input improves tasks like meeting summarization and speaker labeling.'],
 'speakers': ['The first speaker has a clear and informative tone, explaining the limitations of the old pipeline and the benefits of the new direct audio input models. They contribute technical insights and examples of how the new methods work.',
  'The second speaker agrees with the first, adding that the new models can pick up on nuances like who sounded uncertain, and mentions that speaker labeling is now built into some

We can access individual fields from the extracted data:

In [11]:
print("Topic:", audio_data["topic"])
print("\nSummary:", audio_data["summary"])
print("\nKey points:")
for point in audio_data["key_points"]:
    print(" -", point)

Topic: Advancements in AI and Audio Processing

Summary: The conversation discusses the evolution of AI in audio processing, highlighting the shift from transcription-based methods to direct audio input models, which preserve more nuanced information and improve tasks like meeting summarization.

Key points:
 - Traditional pipeline involved transcribing audio first, losing tone and emphasis.
 - New models can take audio directly as input, skipping transcription.
 - Direct audio input improves tasks like meeting summarization and speaker labeling.


#### Considerations:
- **Single-call extraction**: This replaces a transcribe → LLM-summarize pipeline. Fewer API calls, lower latency, lower cost.
- **Schema design**: Keep the JSON schema simple and explicit in the system prompt. Describe each field clearly — the model follows natural language descriptions well.
- **response_format**: Setting `{"type": "json_object"}` guarantees the output is valid JSON. Always pair it with an explicit instruction to return JSON in your prompt.

## Speaker Diarization

With `diarize=True`, the transcription endpoint returns speaker-labeled segments — identifying *who said what* without any external tooling. Each segment includes a `speaker_id`, the text, and timestamps.

This is useful for meeting notes, call center analytics, interview transcripts, and any multi-speaker scenario.

In [12]:
with open(AUDIO_FILE, "rb") as f:
    diarized = client.audio.transcriptions.complete(
        model="voxtral-mini-latest",
        file={"content": f, "file_name": AUDIO_FILE},
        diarize=True,
        timestamp_granularities=["segment"],
    )

for segment in diarized.segments:
    print(f"[{segment.start:.1f}s -> {segment.end:.1f}s] {segment.speaker_id}: {segment.text}")

[0.1s -> 4.3s] speaker_1: So let's talk about where AI and audio are actually heading.
[5.3s -> 8.2s] speaker_1:  A year ago, the pipeline was always the same.
[8.7s -> 12.1s] speaker_1:  Transcribe first, then pass the text to a language model.
[12.1s -> 14.2s] speaker_2:  But it lost a lot along the way.
[14.6s -> 17.7s] speaker_2:  Tone, hesitation, emphasis.
[18.2s -> 20.8s] speaker_2:  All of that just disappears the moment you convert speech to text.
[20.8s -> 25.6s] speaker_1:  And what's exciting now is that some models can take audio directly as input.
[26.3s -> 29.4s] speaker_1:  You skip the transcription step entirely and ask questions straight away.
[29.4s -> 32.0s] speaker_2:  It changes a lot for things like meeting summarization.
[32.7s -> 40.5s] speaker_2:  Instead of running Whisper, cleaning the transcript, then prompting a model, you just send the recording and ask for the action.
[40.5s -> 45.0s] speaker_1:  And the model actually heard the meeting, so it can pick 

We can also reformat the output grouped by speaker:

In [13]:
from collections import defaultdict

speaker_lines = defaultdict(list)
for segment in diarized.segments:
    speaker_lines[segment.speaker_id].append(segment.text.strip())

for speaker, lines in speaker_lines.items():
    print(f"\n{speaker}:")
    for line in lines:
        print(f"  {line}")


speaker_1:
  So let's talk about where AI and audio are actually heading.
  A year ago, the pipeline was always the same.
  Transcribe first, then pass the text to a language model.
  And what's exciting now is that some models can take audio directly as input.
  You skip the transcription step entirely and ask questions straight away.
  And the model actually heard the meeting, so it can pick up on who sounded uncertain.
  The APIs are finally catching up to what people...

speaker_2:
  But it lost a lot along the way.
  Tone, hesitation, emphasis.
  All of that just disappears the moment you convert speech to text.
  It changes a lot for things like meeting summarization.
  Instead of running Whisper, cleaning the transcript, then prompting a model, you just send the recording and ask for the action.
  It's also diarization now, built into some of these APIs.
  Speaker labeling without any external...
  All right, that's a wrap for today.
  Thanks for listening.


#### Considerations:
- **Speaker labels**: Voxtral returns generic labels (`speaker_0`, `speaker_1`, ...). If you need named speakers, follow up with a Q&A call asking the model to identify names from context in the audio.
- **Timestamps**: Combine `diarize=True` with `timestamp_granularities=["segment"]` to get both speaker labels and time boundaries per segment. Note that `timestamp_granularities` is incompatible with the `language` parameter.
- **Realtime limitation**: Diarization is only available on the offline endpoint — it is not compatible with the Realtime API.

## Choosing the Right Model and Endpoint

Voxtral comes in two models, each pairing naturally with a different endpoint and use case:

| | `voxtral-mini-latest` | `voxtral-small-latest` |
|---|---|---|
| **Size** | 3B | 24B |
| **Best endpoint** | `/v1/audio/transcriptions` | Chat completions |
| **Use for** | Transcription, diarization, batch | Q&A, structured extraction, summarization |
| **Cost** | Lower | Higher |
| **Capability** | Text output only | Full audio understanding |

**Rule of thumb:**
- If you need **text out** → use `voxtral-mini-latest` on the transcription endpoint. Fastest and cheapest.
- If you need to **reason about** the audio (questions, summaries, JSON extraction) → use `voxtral-small-latest` via chat completions.

The sections above demonstrate both: transcription and diarization use Mini on the dedicated endpoint; Q&A and structured extraction use Small via chat completions.

---

*Category: Audio | Models: Voxtral Small, Voxtral Mini*